In [ ]:
import os
import cv2
import time
import threading
import ipywidgets as widgets
from IPython.display import display

# ==========================================
# 📁 CONFIGURATION: Set your folder path here
# ==========================================
SAVE_FOLDER = "./my_photos"  # You can change this to any folder path, e.g., "/home/user/Pictures/webcam"

# Ensure the output directory exists
os.makedirs(SAVE_FOLDER, exist_ok=True)

# UI Controls
img_widget = widgets.Image(format='jpeg', width=640, height=480)
btn_capture = widgets.Button(description="📸 Capture Frame", button_style='primary')
btn_retake = widgets.Button(description="🔄 Retake", button_style='warning', disabled=True)
btn_save = widgets.Button(description="💾 Save Photo", button_style='success', disabled=True)
status_label = widgets.Label(value=f"Status: Live Feed Running... (Saving to: '{SAVE_FOLDER}')")

controls = widgets.HBox([btn_capture, btn_retake, btn_save])
display(widgets.VBox([img_widget, controls, status_label]))

# Webcam state
cap = cv2.VideoCapture(0)
captured_frame = None
is_frozen = False
is_running = True

def update_feed():
    global captured_frame, is_frozen, is_running
    while is_running and cap.isOpened():
        if not is_frozen:
            ret, frame = cap.read()
            if ret:
                # Mirror horizontally for natural camera view
                frame = cv2.flip(frame, 1)
                captured_frame = frame.copy()
                
                # Convert BGR to JPEG for widget display
                _, buffer = cv2.imencode('.jpg', frame)
                img_widget.value = buffer.tobytes()
        time.sleep(0.03)  # ~30 FPS preview

# Start background thread for live preview
thread = threading.Thread(target=update_feed, daemon=True)
thread.start()

# Button Callbacks
def on_capture_clicked(b):
    global is_frozen
    is_frozen = True
    btn_capture.disabled = True
    btn_retake.disabled = False
    btn_save.disabled = False
    status_label.value = "Status: Frame frozen! Click 'Save' to keep it or 'Retake' to try again."

def on_retake_clicked(b):
    global is_frozen
    is_frozen = False
    btn_capture.disabled = False
    btn_retake.disabled = True
    btn_save.disabled = True
    status_label.value = "Status: Live Feed Running..."

def on_save_clicked(b):
    global captured_frame, is_frozen
    if captured_frame is not None:
        # Generate a unique timestamped filename (YYYYMMDD_HHMMSS)
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        filename = f"photo_{timestamp}.jpg"
        filepath = os.path.join(SAVE_FOLDER, filename)
        
        # Save photo to specified directory
        cv2.imwrite(filepath, captured_frame)
        status_label.value = f"Status: Saved -> '{filepath}'. Ready for next photo!"
        
        # Resume live feed so you can take more photos without restarting the cell
        is_frozen = False
        btn_capture.disabled = False
        btn_retake.disabled = True
        btn_save.disabled = True

btn_capture.on_click(on_capture_clicked)
btn_retake.on_click(on_retake_clicked)
btn_save.on_click(on_save_clicked)